# DFIR-Guardrail Demo
This Colab notebook demonstrates the DFIR-Guardrail pipeline on a free T4 GPU. It clones the GitHub repository, installs Ollama natively on the Colab VM, pulls the model, and executes the evaluation pipeline.

In [ ]:
# 1. Install Dependencies and Ollama
!sudo apt-get update -y
# Added pciutils and lshw so Ollama detects the Colab T4 GPU
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import requests

print("Starting Ollama daemon...")
ollama_process = subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for API to be ready
for i in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            print("Ollama daemon is ready!")
            break
    except:
        time.sleep(1)
else:
    print("Warning: Ollama daemon did not become ready within 30 seconds.")

print("Pulling phi3 model (this may take a minute)...")
!/usr/local/bin/ollama pull phi3

In [ ]:
# 2. Clone Repository & Install Dependencies
!git clone https://github.com/YedidyaBarGad/DFIR-Guardrail.git
%cd DFIR-Guardrail

!pip install -r requirements.txt

In [ ]:
# 3. Execute the Pipeline
!PYTHONPATH=. python src/pipeline.py --num_samples 5000

In [ ]:
# 4. Cleanup
ollama_process.terminate()
print("Ollama daemon terminated.")